# SC Linac overlays

This creates the overlays for L1, L1H, L2, L3. 



In [1]:
from pytao import Tao
from pprint import pprint
import json
import os
import numpy as np
from math import sqrt, cos, pi

In [2]:
# Note: comment out:
#    call, file = $LCLS_LATTICE/bmad/overlays/sc/linac_overlays.bmad
# in sc_sxr.lat.bmad
# This will pick up the design 
    
M = Tao('-init $LCLS_LATTICE/bmad/models/sc_hxr/tao.init -noplot')

In [3]:
from pytao.tao_ctypes.util import parse_bool, parse_tao_lat_ele_list

In [4]:
SLIST = M.lat_list('*', 'ele.s', flags='-array_out -no_slaves')
LLIST = M.lat_list('*', 'ele.l', flags='-array_out -no_slaves')
NAMES = M.lat_list('*', 'ele.name', flags='-no_slaves')

for s, l, n in zip(SLIST[0:20], LLIST[0:20], NAMES[0:20]):
    print (n, l, s)

BEGINNING 0.0 0.0
BEGGUNB 0.0 0.0
DGBCA -0.071705 -0.071705
SOL1BKB 0.0 -0.071705
DGBCB 0.071705 0.0
CATHODEB 0.0 0.0
DGUN 0.04 0.04
DG001 0.16348 0.20348
SOL1B 0.0861 0.28957999999999995
SQ01B 0.0 0.24652999999999997
CQ01B 0.0 0.24652999999999997
DG002 0.09789999999999999 0.38747999999999994
VV01B 0.0 0.38747999999999994
DG003 0.09702 0.48449999999999993
XC01B 0.0 0.48449999999999993
YC01B 0.0 0.48449999999999993
DG004 0.00515 0.4896499999999999
BPM1B 0.0 0.4896499999999999
DG005 0.105042 0.5946919999999999
IM01B 0.0 0.5946919999999999


In [5]:
# Index lookup function
ix_of = parse_tao_lat_ele_list(M.cmd('python lat_ele_list 1@0'))
s_of = {}
for n,s in zip(NAMES, SLIST):
    s_of[n] = s

In [6]:
CAVS = [name for name in NAMES if name.startswith('CAV') and ('#' not in name)]
KLYS = {}
for cav in CAVS:
    name, section = cav[0:-1], cav[-1:]
    if name not in KLYS:
        KLYS[name] = []
    KLYS[name].append(section)
len(CAVS)

296

# Linac overlays

L1 is CM 02 - 03

L2 is CM 04 - 15

L3 is CM 16 - 35

prefix: CAVL
Each CM has cavities like CAVL157 for CM 15, Cavity 7. 


3.9 GHz cavities
L1H is CMH1 - CMH2

prefix: CAVC

In [7]:
for i in range(10):
    print(f'{i:02d}') 

00
01
02
03
04
05
06
07
08
09


In [8]:
def CM_eles(ix, prefix='CAVL'):
    """Get eles for cyromodule"""
    return [f'{prefix}{ix:02d}{i}' for i in range(1,9)]
CM_eles(22)    

['CAVL221',
 'CAVL222',
 'CAVL223',
 'CAVL224',
 'CAVL225',
 'CAVL226',
 'CAVL227',
 'CAVL228']

In [9]:
# Define eles by linac
def Linac_eles(i1, i2, prefix='CAVL'):
    eles = []
    for i in range(i1, i2+1):
        eles += CM_eles(i, prefix=prefix)
    return eles

L0 = Linac_eles(1,1)
L1 = Linac_eles(2,3)
L2 = Linac_eles(4,15)
L3 = Linac_eles(16, 35)
L1H = Linac_eles(1,2, prefix='CAVC')

In [10]:
# Make overlays
# Voltage actually controls gradient
def phase_overlay(name, cavs):
    ncav = len(cavs)
    lines = []
    lines.append('!--------------\n')
    lines.append('! Linac phase and voltage overlay\n')
    lines.append(f'{name}: overlay = {{\n')
    i=0
    for n in cavs:
        i += 1
        lines.append(f'  {n}[phi0]:phase_deg/360, {n}[gradient]:voltage/{n}[L]/{ncav},')
        if i == 2:
            i = 0
            lines.append('\n')   
    if lines[-1] == '\n':
        lines.pop()
        
    lines[-1] = lines[-1][:-1]+'}, var = {phase_deg, voltage}\n\n'
    return ''.join(lines)
print(phase_overlay('SC_L1',L1)           )

!--------------
! Linac phase and voltage overlay
SC_L1: overlay = {
  CAVL021[phi0]:phase_deg/360, CAVL021[gradient]:voltage/CAVL021[L]/16,  CAVL022[phi0]:phase_deg/360, CAVL022[gradient]:voltage/CAVL022[L]/16,
  CAVL023[phi0]:phase_deg/360, CAVL023[gradient]:voltage/CAVL023[L]/16,  CAVL024[phi0]:phase_deg/360, CAVL024[gradient]:voltage/CAVL024[L]/16,
  CAVL025[phi0]:phase_deg/360, CAVL025[gradient]:voltage/CAVL025[L]/16,  CAVL026[phi0]:phase_deg/360, CAVL026[gradient]:voltage/CAVL026[L]/16,
  CAVL027[phi0]:phase_deg/360, CAVL027[gradient]:voltage/CAVL027[L]/16,  CAVL028[phi0]:phase_deg/360, CAVL028[gradient]:voltage/CAVL028[L]/16,
  CAVL031[phi0]:phase_deg/360, CAVL031[gradient]:voltage/CAVL031[L]/16,  CAVL032[phi0]:phase_deg/360, CAVL032[gradient]:voltage/CAVL032[L]/16,
  CAVL033[phi0]:phase_deg/360, CAVL033[gradient]:voltage/CAVL033[L]/16,  CAVL034[phi0]:phase_deg/360, CAVL034[gradient]:voltage/CAVL034[L]/16,
  CAVL035[phi0]:phase_deg/360, CAVL035[gradient]:voltage/CAVL035[L]/16,  

In [11]:
len(L1)

16

# Get defaults

In [12]:
#dat = M.cmd('python ele:gen_attribs 1@0>>1491|model')

params = M.ele_gen_attribs(1491)
#params = tao_parameter_dict(dat)
params

{'L': 0.0,
 'units#L': 'm',
 'TILT': 0.0,
 'units#TILT': 'rad',
 'X_GAIN_ERR': 0.0,
 'units#X_GAIN_ERR': 'm',
 'Y_GAIN_ERR': 0.0,
 'units#Y_GAIN_ERR': 'm',
 'CRUNCH': 0.0,
 'units#CRUNCH': 'rad',
 'NOISE': 0.0,
 'units#NOISE': '',
 'OSC_AMPLITUDE': 0.0,
 'units#OSC_AMPLITUDE': 'm',
 'X_GAIN_CALIB': 0.0,
 'units#X_GAIN_CALIB': 'm',
 'Y_GAIN_CALIB': 0.0,
 'units#Y_GAIN_CALIB': 'm',
 'CRUNCH_CALIB': 0.0,
 'units#CRUNCH_CALIB': 'rad',
 'X_OFFSET_CALIB': 0.0,
 'units#X_OFFSET_CALIB': 'm',
 'Y_OFFSET_CALIB': 0.0,
 'units#Y_OFFSET_CALIB': 'm',
 'TILT_CALIB': 0.0,
 'units#TILT_CALIB': 'rad',
 'DE_ETA_MEAS': 0.0,
 'units#DE_ETA_MEAS': '',
 'N_SAMPLE': 0.0,
 'units#N_SAMPLE': '',
 'X_DISPERSION_ERR': 0.0,
 'units#X_DISPERSION_ERR': 'm',
 'Y_DISPERSION_ERR': 0.0,
 'units#Y_DISPERSION_ERR': 'm',
 'X_DISPERSION_CALIB': 0.0,
 'units#X_DISPERSION_CALIB': 'm',
 'Y_DISPERSION_CALIB': 0.0,
 'units#Y_DISPERSION_CALIB': 'm',
 'X_PITCH': 0.0,
 'units#X_PITCH': '',
 'Y_PITCH': 0.0,
 'units#Y_PITCH': '',
 'X

In [13]:
# Get all parameters for cavities
CAVDAT = {}
for name in CAVS:
    ix = ix_of[name]
    #print(name, ix)
    CAVDAT[name] =  M.ele_gen_attribs(ix)
#CAVDAT['CAV318']['VOLTAGE'].value

In [14]:
def defaults(eles, param='VOLTAGE'):
    vals = list(set(CAVDAT[C.upper()][param.upper()] for C in eles))
    assert len(vals) == 1
    return vals[0]
defaults(L1, 'VOLTAGE'), defaults(L1, 'phi0')      
defaults(L1H, 'gradient'), defaults(L1H, 'phi0')      

(10840833.0939399, -0.479166666666667)

In [15]:
CAVDAT['CAVL021']['PHI0']

-0.0691666666666667

In [16]:
set([CAVDAT[C]['VOLTAGE'] for C in L1]), set([CAVDAT[C]['VOLTAGE'] for C in L2]), set([CAVDAT[C]['VOLTAGE'] for C in L3])

({14434711.0215866}, {15404496.9517917}, {15625000.0})

In [17]:

L1_voltage = defaults(L1, 'voltage')*len(L1)
L1_phase = defaults(L1, 'phi0')*2*pi
L1_dE = L1_voltage * cos( L1_phase)            
    
L1H_voltage = defaults(L1H, 'voltage')*len(L1H)
L1H_phase   = defaults(L1H, 'phi0')*2*pi  
L1H_dE = L1H_voltage * cos( L1H_phase)   
    
L2_voltage = defaults(L2, 'voltage')*len(L2)
L2_phase   = defaults(L2, 'phi0')*2*pi    
L2_dE = L2_voltage * cos( L2_phase)     
    
L3_voltage = defaults(L3, 'voltage')*len(L3)
L3_phase   = defaults(L3, 'phi0')*2*pi     
L3_dE = L3_voltage * cos( L3_phase) 

DEFAULTS=f"""
! Design linac phasing and voltages

SC_L1[voltage]   = {L1_voltage}
SC_L1[phase_deg] = {np.round(L1_phase * 180/pi, 9)}

SC_L1H[voltage]   = {L1H_voltage}
SC_L1H[phase_deg] = {np.round(L1H_phase * 180/pi, 9)}

SC_L2[voltage]   = {L2_voltage}
SC_L2[phase_deg] = {np.round(L2_phase * 180/pi, 9)}

SC_L3[voltage]   = {L3_voltage}
SC_L3[phase_deg] = {np.round(L3_phase * 180/pi, 9)}

"""
print(DEFAULTS)


! Design linac phasing and voltages

SC_L1[voltage]   = 230955376.3453856
SC_L1[phase_deg] = -24.9

SC_L1H[voltage]   = 60000000.0
SC_L1H[phase_deg] = -172.5

SC_L2[voltage]   = 1478831707.372003
SC_L2[phase_deg] = -32.3

SC_L3[voltage]   = 2500000000.0
SC_L3[phase_deg] = 0.0




In [18]:
lines = []
for c in L1+L1H+L2+L3:
    lines.append(f'{c}[field_master]=T')
FIELD_MASTER = '\n'.join(lines)

In [19]:
# Energy profile
dE = [100e6, L1_dE, L1H_dE, L2_dE , L3_dE]
np.cumsum(dE)/1e6

array([ 100.        ,  309.48669168,  250.        , 1500.        ,
       4000.        ])

# Write

In [20]:
with open('../linac_overlays.bmad', 'w') as f:
    #f.write(FIELD_MASTER+'\n')
    f.write(phase_overlay('SC_L1',L1))
    f.write(phase_overlay('SC_L1H',L1H))
    f.write(phase_overlay('SC_L2',L2))
    f.write(phase_overlay('SC_L3',L3))
    f.write(DEFAULTS)   
    
    